# Lab 05 · Shuffle and joins: how Spark picks a join

**Part A** (about 45 min): measure shuffles, predict Spark's join choice, force other joins, and join without a shuffle.<br>
**Part B** (about 75 min): 13 join problems: semi, anti, full, self, range and cross joins.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>3.1</b> What a shuffle is · <b>3.2</b> When a broadcast join is triggered · <b>3.3</b> The cost of a shuffle · <b>3.4</b> Avoiding a shuffle in a join · <b>3.5</b> Bucketing · <b>3.6</b> Sort-merge join stages · <b>3.7</b> Hash join · <b>3.8</b> Cartesian product · <b>3.9</b> How Spark chooses a join · <b>3.10</b> AQE and joins · <b>3.12</b> Broadcast hash join vs. shuffle hash join</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`.
* **Part A**: run the code under the question (or add a line), look at the output or the plan, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Shuffle operations](https://spark.apache.org/docs/latest/rdd-programming-guide.html#shuffle-operations): what a shuffle does and costs
* [Join strategy hints](https://spark.apache.org/docs/latest/sql-performance-tuning.html#join-strategy-hints): `broadcast`, `shuffle_hash`, `merge`
* [Automatically broadcasting joins](https://spark.apache.org/docs/latest/sql-performance-tuning.html#automatically-broadcasting-joins): `spark.sql.autoBroadcastJoinThreshold`
* [AQE: converting sort-merge join to broadcast join](https://spark.apache.org/docs/latest/sql-performance-tuning.html#converting-sort-merge-join-to-broadcast-join)
* [Bucketing (DataFrameWriter.bucketBy)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.bucketBy.html)
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html): join types and conditions
* [JOIN (SQL reference)](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html): semi, anti, cross joins explained
* [DataFrame.alias](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.alias.html): naming the two sides of a self join

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also makes:
* **big_orders** (Part A): 2,000,000 generated orders: *order_id*, *customer_id* (500,000 customers), *status*, *quantity*
* **payments** (Part A): 2,000,000 rows: *order_id*, *amount* (0 to 999)
* **promotions** (Part B): *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*

## Setup
Run this cell once. It starts Spark, saves the three shop tables as Parquet files in `data/shop/`, and defines *big_orders*, *payments*, *promotions* and the helpers `noop` and `shuffle_mb`.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab05")
         .config("spark.sql.adaptive.enabled", "false")        # AQE starts off; question A6 turns it on
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.warehouse.dir", os.path.abspath("data/shop/warehouse"))   # tables of question A5
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def noop(df):
    """Runs the whole query but keeps no result (Spark's "noop" sink)."""
    df.write.format("noop").mode("overwrite").save()

import json, urllib.request

def spark_api(path):
    """Reads the Spark UI's REST API, e.g. spark_api("/stages")."""
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

def shuffle_mb(action):
    """Runs action() and returns how many MB its stages wrote to the shuffle."""
    before = {s["stageId"] for s in spark_api("/stages")}
    action()
    time.sleep(0.5)
    new_stages = [s for s in spark_api("/stages") if s["stageId"] not in before]
    return round(sum(s["shuffleWriteBytes"] for s in new_stages) / 1e6, 1)

def pick(salt, n):
    """A number from 0 to n-1 computed from the row id: looks random, but is the same on every machine."""
    return F.pmod(F.xxhash64("id", F.lit(salt)), F.lit(n))

# Part A: two big generated tables (nothing is computed until you use them)
big_orders = spark.range(2_000_000).select(
    (F.col("id") + 1).alias("order_id"),
    (pick(1, 500_000) + 1).alias("customer_id"),
    F.element_at(F.array(F.lit("delivered"), F.lit("shipped"), F.lit("cancelled")), (pick(2, 3) + 1).cast("int")).alias("status"),
    (pick(3, 5) + 1).alias("quantity"))
payments = spark.range(2_000_000).select((F.col("id") + 1).alias("order_id"), pick(4, 1000).alias("amount"))

# Part B: promotions (a discount on one category for some days)
promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date, discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
promotions.show()
big_orders.show(3)
payments.show(3)

## Part A · Shuffles and joins

### What a shuffle costs <span style="color:#d93025">(3.1, 3.3)</span>

**A1. How much data does each shuffle write: grouping `big_orders` by customer_id, grouping it by status, and `big_orders.repartition(8)`? Measure each with `shuffle_mb` (example in the cell).**

In [ ]:
# My answer:
print("group by customer_id:", shuffle_mb(lambda: noop(big_orders.groupBy("customer_id").count())), "MB")

### Which join does Spark pick? <span style="color:#d93025">(3.2, 3.8, 3.9)</span>

**A2. Predict the join algorithm of each query, then check with `explain()`: (a) big_orders with customers, (b) big_orders with payments, (c) customers crossed with products.**

In [ ]:
# My guess: a = ..., b = ..., c = ...
a = big_orders.join(customers, "customer_id")
b = big_orders.join(payments, "order_id")
c = customers.crossJoin(products)

**A3. Spark broadcasts a table when its estimated size is under 10 MB. What size does Spark estimate for customers and for big_orders? Look for `Statistics` in `explain("cost")`.**

In [ ]:
# My answer:
customers.explain("cost")

**A4. Force each join algorithm with a hint: `customers.hint("broadcast")`, `"shuffle_hash"` and `"merge"`. Which ones shuffle big_orders? Look at `explain()` too: which one also sorts?**

In [ ]:
# My answer:
for hint in ["broadcast", "shuffle_hash", "merge"]:
    q = big_orders.join(customers.hint(hint), "customer_id")
    print(f"{hint:12}", shuffle_mb(lambda: noop(q)), "MB shuffled")

### Joining without a shuffle <span style="color:#d93025">(3.4, 3.5)</span>

**A5. Save big_orders and payments as tables bucketed by order_id into 8 buckets, then join them. How many `Exchange`s does the plan have, and how many MB are shuffled? Compare with the same join of the plain DataFrames.**

In [ ]:
# My answer:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)        # no broadcasting: we want to see the shuffle
big_orders.write.mode("overwrite").bucketBy(8, "order_id").sortBy("order_id").saveAsTable("orders_bucketed")
payments.write.mode("overwrite").bucketBy(8, "order_id").sortBy("order_id").saveAsTable("payments_bucketed")
bucketed = spark.table("orders_bucketed").join(spark.table("payments_bucketed"), "order_id")

### AQE and joins <span style="color:#d93025">(3.10)</span>

**A6. Turn AQE on. Join big_orders with the payments of amount 7 (about 2,000 rows). Which join is in the plan before running, and which one after `q.collect()`?**

In [ ]:
# My answer:
spark.conf.set("spark.sql.adaptive.enabled", "true")
q = big_orders.join(payments.filter(F.col("amount") == 7), "order_id")
q.explain()          # before running

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Join problems
13 problems on the shop tables and *promotions*. When both sides of a join have a column with the same name (a self join), name the sides with `.alias("a")` and `.alias("b")` and refer to `F.col("a.city")`.

**B1. Which customers have at least one delivered order? Show customer_id and name (use a `left_semi` join).**

Expected output:

```
+-----------+-------------+
|customer_id|         name|
+-----------+-------------+
|          1| Aziz Karimov|
|          2|Bella Schmidt|
|          3|  Carlos Ruiz|
|          5|  Emma Wilson|
+-----------+-------------+
```

<details><summary>Hint</summary>

A `"left_semi"` join keeps the rows of the left table that have a match, without adding columns or duplicating rows.

</details>

**B2. Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 when none). Products missing from the catalog show an empty name. Columns: product_id, product_name, orders.**

Expected output:

```
+----------+------------+------+
|product_id|product_name|orders|
+----------+------------+------+
|       101|      Laptop|     3|
|       102|  Headphones|     2|
|       103|Coffee Maker|     3|
|       104|     Blender|     2|
|       105|       Novel|     3|
|       106|    Cookbook|     2|
|       107|   Desk Lamp|     0|
|       108|        NULL|     1|
+----------+------------+------+
```

<details><summary>Hint</summary>

A `"full"` join keeps the rows of both sides; then turn the missing counts into 0.

</details>

**B3. Show the pairs of customers who live in the same city: city, customer_a and customer_b (names; each pair once, customer_a with the smaller customer_id).**

Expected output:

```
+--------+-------------+----------------+
|    city|   customer_a|      customer_b|
+--------+-------------+----------------+
|  Berlin|Bella Schmidt|       Grace Lee|
|Tashkent| Aziz Karimov|Dilnoza Rahimova|
+--------+-------------+----------------+
```

<details><summary>Hint</summary>

A self join: the table joined with itself. `a.customer_id < b.customer_id` keeps each pair once and drops pairs of a customer with themself.

</details>

**B4. Which other customers ordered at least one product that customer 1 also ordered? Show their names, A to Z.**

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
|  Carlos Ruiz|
|  Emma Wilson|
| Farid Aliyev|
+-------------+
```

<details><summary>Hint</summary>

First the products of customer 1, then the other customers' orders of those products.

</details>

**B5. Show the pairs of orders by the same customer placed at most 20 days apart: customer_id, first_order, second_order and days_between (first_order is the smaller order_id).**

Expected output:

```
+-----------+-----------+------------+------------+
|customer_id|first_order|second_order|days_between|
+-----------+-----------+------------+------------+
|          1|       1001|        1002|           0|
|          2|       1003|        1005|          11|
|          5|       1008|        1014|          18|
+-----------+-----------+------------+------------+
```

<details><summary>Hint</summary>

Self join on the customer, with two more conditions: `o1.order_id < o2.order_id` and the date difference.

</details>

**B6. For each delivered order, find the promotion of its product's category that was running on the order date. Show order_id, product_name, promo_id and discount_pct; orders without a promotion drop out.**

Expected output:

```
+--------+------------+--------+------------+
|order_id|product_name|promo_id|discount_pct|
+--------+------------+--------+------------+
|    1001|      Laptop|  WINTER|          10|
|    1003|  Headphones|  WINTER|          10|
|    1009|       Novel|BOOKWEEK|          20|
|    1011|     Blender|  SPRING|          15|
|    1013|      Laptop|   FLASH|           5|
+--------+------------+--------+------------+
```

<details><summary>Hint</summary>

A range join: the join condition is an expression, `category` equal and `order_date` between start_date and end_date.

</details>

**B7. Revenue after promotions: for every delivered order, revenue = quantity × price × (1 − discount_pct / 100), without discount when no promotion was running. Show order_id and revenue (2 decimals).**

Expected output:

```
+--------+-------+
|order_id|revenue|
+--------+-------+
|    1001| 1080.0|
|    1002|   45.0|
|    1003|  270.0|
|    1005|   25.0|
|    1007|  150.0|
|    1009|   48.0|
|    1011|   51.0|
|    1013| 1140.0|
|    1014|   75.0|
+--------+-------+
```

<details><summary>Hint</summary>

The same range join as B6, but `left`, so that orders without a promotion stay; a missing discount counts as 0.

</details>

**B8. Make a table with every combination of customers 1 to 3 and every product category, with the number of orders (0 when none). Columns: customer_id, category, orders.**

Expected output:

```
+-----------+-----------+------+
|customer_id|   category|orders|
+-----------+-----------+------+
|          1|      Books|     1|
|          1|Electronics|     2|
|          1|       Home|     0|
|          1|    Kitchen|     1|
|          2|      Books|     1|
|          2|Electronics|     2|
|          2|       Home|     0|
|          2|    Kitchen|     0|
|          3|      Books|     1|
|          3|Electronics|     0|
|          3|       Home|     0|
|          3|    Kitchen|     1|
+-----------+-----------+------+
```

<details><summary>Hint</summary>

`crossJoin` builds all combinations; then left-join the real counts on both columns.

</details>

**B9. Which orders point to a product that is not in products? Show order_id and product_id.**

Expected output:

```
+--------+----------+
|order_id|product_id|
+--------+----------+
|    1016|       108|
+--------+----------+
```

<details><summary>Hint</summary>

`"left_anti"` keeps the rows without a match.

</details>

**B10. Show the revenue of delivered orders per city and category (customers without a city drop out). Columns: city, category, revenue.**

Expected output:

```
+--------+-----------+-------+
|    city|   category|revenue|
+--------+-----------+-------+
|  Berlin|      Books|   25.0|
|  Berlin|Electronics| 1500.0|
|  London|      Books|   75.0|
|  Madrid|      Books|   60.0|
|Tashkent|      Books|   45.0|
|Tashkent|Electronics| 1350.0|
+--------+-----------+-------+
```

<details><summary>Hint</summary>

Join three tables, then group by two columns.

</details>

**B11. Which customers ordered in all three months (January, February and March)? Show their names.**

Expected output:

```
+-------------+
|         name|
+-------------+
| Aziz Karimov|
|Bella Schmidt|
+-------------+
```

<details><summary>Hint</summary>

Count the different months per customer and keep those with 3.

</details>

**B12. Which customers had all of their orders delivered (and at least one order)? Show their names.**

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
+-------------+
```

<details><summary>Hint</summary>

Customers with orders (`left_semi`) minus customers with a non-delivered order (`left_anti`).

</details>

**B13. For each order placed in March, show order_id and the customer's name, or `Guest` when the order has no customer.**

Expected output:

```
+--------+----------------+
|order_id|        customer|
+--------+----------------+
|    1008|     Emma Wilson|
|    1009|     Carlos Ruiz|
|    1010|Dilnoza Rahimova|
|    1011|           Guest|
|    1012|    Farid Aliyev|
|    1013|   Bella Schmidt|
|    1014|     Emma Wilson|
|    1015|    Aziz Karimov|
|    1016|     Carlos Ruiz|
+--------+----------------+
```

<details><summary>Hint</summary>

A `left` join keeps the guest order; `F.coalesce` replaces its missing name.

</details>

---
**Done?** In the Theory part of Day 5, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;How much data does each shuffle write: grouping big_orders by customer_id, grouping it by status, and ...</summary>

**About 11 MB by customer_id, almost nothing by status, and about 39 MB for `repartition(8)`.** A shuffle writes each row to local disk, sorted by its target partition; the next stage then fetches those blocks over the network. So a shuffle costs serialization, disk, network, and a stage boundary where all tasks wait. `groupBy` first aggregates inside each partition (partial aggregation), so only one row per key and partition is shuffled: 3 statuses make a few rows, 500,000 customers make most of the rows. `repartition` moves every full row. The cost grows with the bytes that move: filter rows and select columns before a shuffle.

```python
print("group by customer_id:", shuffle_mb(lambda: noop(big_orders.groupBy("customer_id").count())), "MB")
print("group by status:     ", shuffle_mb(lambda: noop(big_orders.groupBy("status").count())), "MB")
print("repartition(8):      ", shuffle_mb(lambda: noop(big_orders.repartition(8))), "MB")
```

Result:

```
group by customer_id: 11.5 MB
group by status:      0.0 MB
repartition(8):       39.1 MB
```

</details>

<details>
<summary><b>A2</b> &nbsp;Predict the join algorithm of each query, then check with explain(): (a) big_orders with customers, (b) ...</summary>

**(a) BroadcastHashJoin, (b) SortMergeJoin, (c) BroadcastNestedLoopJoin.** Spark decides in this order: a join hint wins; otherwise, if one side is small enough to broadcast (estimated under `spark.sql.autoBroadcastJoinThreshold`, 10 MB), it uses a broadcast hash join (a); two big tables joined on an equal key get a sort-merge join by default (b); a join without an equality condition can't hash or sort on a key, so it is a nested loop join, with the small side broadcast (c). If neither side of such a join can be broadcast, it becomes a **Cartesian product**: every row paired with every row (7 × 7 = 49 rows here, but 1 million × 1 million would be 10^12).

```python
a = big_orders.join(customers, "customer_id")
b = big_orders.join(payments, "order_id")
c = customers.crossJoin(products)
for q in [a, b, c]:
    q.explain()
```

Result:

```
+- *(2) BroadcastHashJoin [customer_id#32L], [cast(customer_id#15 as bigint)], Inner, BuildRight, false, false
+- *(5) SortMergeJoin [order_id#31L], [order_id#36L], Inner
*(2) BroadcastNestedLoopJoin BuildRight, Cross
```

</details>

<details>
<summary><b>A3</b> &nbsp;Spark broadcasts a table when its estimated size is under 10 MB. What size does Spark estimate for customers ...</summary>

**About 1.8 KB for customers (the size of its file) and about 50 MiB for big_orders** (2,000,000 rows × the estimated width of a row). Only customers is under the threshold, so it is the side Spark broadcasts. The decision rests on *estimates*: without statistics Spark cannot know how small a filtered table really is (A6), which is why `ANALYZE TABLE` statistics and AQE matter.

```python
customers.explain("cost")
big_orders.explain("cost")
```

Result:

```
Relation [customer_id#15,name#16,email#17,city#18,signup_date#19] parquet, Statistics(sizeInBytes=1809.0 B)
Project [(id#30L + 1) AS order_id#31L, (pmod(xxhash64(id#30L, 1, 42), 500000) + 1) AS customer_id#32L, element_at([delivered,shipped,cancelled], cast((pmod(xxhash64(id#30L, 2, 42), 3) + 1) as int), None, true) AS status#33, (pmod(xxhash64(id#30L, 3, 42), 5) + 1) AS quantity#34L], Statistics(sizeInBytes=49.6 MiB)
+- Range (0, 2000000, step=1, splits=Some(4)), Statistics(sizeInBytes=15.3 MiB, rowCount=2.00E+6)
```

</details>

<details>
<summary><b>A4</b> &nbsp;Force each join algorithm with a hint: customers.hint("broadcast"), "shuffle_hash" and "merge". Which ones ...</summary>

**broadcast: no shuffle at all; shuffle_hash and merge: both sides shuffled (about 37 MB); merge also sorts.**
* **Broadcast hash join**: the small table is sent whole to every executor and built into a hash table; each task probes it with its own rows of big_orders. The big side never moves: the fastest join, if the small side fits in memory.
* **Shuffle hash join**: both sides are shuffled by the key; each task builds a hash table from its part of the smaller side and probes it with the other side. No sort, but the hash table must fit in the task's memory.
* **Sort-merge join**: both sides are shuffled by the key and sorted; then each task walks through both sorted lists at once, like a zipper. Its stages: shuffle-write side A, shuffle-write side B, then sort and merge. Sorting can spill to disk, so it works for any size: the default for two big tables.

**Hash join** in short: build a hash table on the join key from one side, then look up every row of the other side in it.

```python
for hint in ["broadcast", "shuffle_hash", "merge"]:
    q = big_orders.join(customers.hint(hint), "customer_id")
    print(f"{hint:12}", shuffle_mb(lambda: noop(q)), "MB shuffled")
big_orders.join(customers.hint("merge"), "customer_id").explain()
```

Result:

```
broadcast    0.0 MB shuffled
shuffle_hash 37.6 MB shuffled
merge        37.6 MB shuffled
+- *(5) SortMergeJoin [customer_id#32L], [cast(customer_id#15 as bigint)], Inner
   :- *(2) Sort [customer_id#32L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#32L, 200), ENSURE_REQUIREMENTS, [plan_id=656]
   +- *(4) Sort [cast(customer_id#15 as bigint) ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(cast(customer_id#15 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=665]
```

</details>

<details>
<summary><b>A5</b> &nbsp;Save big_orders and payments as tables bucketed by order_id into 8 buckets, then join them. How many ...</summary>

**No `Exchange` and 0 MB, against about 58 MB for the plain DataFrames.** **Bucketing** writes a table pre-shuffled: every row is hashed by order_id into one of 8 bucket files, so bucket 3 of one table can only match bucket 3 of the other. When both tables are bucketed the same way on the join key, Spark joins them bucket by bucket, with no shuffle (and, thanks to `sortBy`, no sort either). Other ways to avoid a shuffle in a join: broadcast the small side, or reuse data that is already partitioned by the key.

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)        # no broadcasting: we want to see the shuffle
big_orders.write.mode("overwrite").bucketBy(8, "order_id").sortBy("order_id").saveAsTable("orders_bucketed")
payments.write.mode("overwrite").bucketBy(8, "order_id").sortBy("order_id").saveAsTable("payments_bucketed")
bucketed = spark.table("orders_bucketed").join(spark.table("payments_bucketed"), "order_id")
bucketed.explain()
print("bucketed:", shuffle_mb(lambda: noop(bucketed)), "MB shuffled")
print("plain:   ", shuffle_mb(lambda: noop(big_orders.join(payments, "order_id"))), "MB shuffled")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
```

Result:

```
+- *(3) SortMergeJoin [order_id#146L], [order_id#150L], Inner
   :        +- FileScan parquet spark_catalog.default.orders_bucketed[order_id#146L,customer_id#147L,status#148,quantity#149L] Batched: true, Bucketed: true, DataFilters: [isnotnull(order_id#146L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(order_id)], ReadSchema: struct<order_id:bigint,customer_id:bigint,status:string,quantity:bigint>, SelectedBucketsCount: 8 out of 8
            +- FileScan parquet spark_catalog.default.payments_bucketed[order_id#150L,amount#151L] Batched: true, Bucketed: true, DataFilters: [isnotnull(order_id#150L)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(order_id)], ReadSchema: struct<order_id:bigint,amount:bigint>, SelectedBucketsCount: 8 out of 8
bucketed: 0.0 MB shuffled
plain:    58.4 MB shuffled
```

</details>

<details>
<summary><b>A6</b> &nbsp;Turn AQE on. Join big_orders with the payments of amount 7 (about 2,000 rows). Which join is in the plan ...</summary>

**SortMergeJoin before, BroadcastHashJoin after.** Before running, Spark can't know how many payments have amount 7, so it estimates the filtered side as big. AQE runs the shuffle stages first, sees that the filtered side is only a few KB, and switches the join to a broadcast hash join at runtime (`AQEShuffleRead local` reads the already shuffled data without another shuffle). AQE also coalesces partitions (Lab 03) and splits skewed partitions in joins (Lab 06).

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
q = big_orders.join(payments.filter(F.col("amount") == 7), "order_id")
q.explain()          # before running
q.collect()
q.explain()          # after running
```

Result:

```
AdaptiveSparkPlan isFinalPlan=false
   +- SortMergeJoin [order_id#31L], [order_id#36L], Inner
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
      +- *(3) BroadcastHashJoin [order_id#31L], [order_id#36L], Inner, BuildRight, false, false
+- == Initial Plan ==
   +- SortMergeJoin [order_id#31L], [order_id#36L], Inner
```

</details>

<details>
<summary><b>B1</b> &nbsp;Which customers have at least one delivered order? Show customer_id and name (use a left_semi join).</summary>

```python
(customers.join(orders.filter(F.col("status") == "delivered"), "customer_id", "left_semi")
          .select("customer_id", "name")
          .orderBy("customer_id")
          .show())
```

</details>

<details>
<summary><b>B2</b> &nbsp;Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 ...</summary>

```python
counts = orders.groupBy("product_id").agg(F.count("*").alias("orders"))
(products.join(counts, "product_id", "full")
         .select("product_id", "product_name", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
         .orderBy("product_id")
         .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;Show the pairs of customers who live in the same city: city, customer_a and customer_b (names; each pair ...</summary>

```python
a, b = customers.alias("a"), customers.alias("b")
(a.join(b, (F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")))
  .select(F.col("a.city").alias("city"), F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
  .orderBy("city")
  .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;Which other customers ordered at least one product that customer 1 also ordered? Show their names, A to Z.</summary>

```python
aziz_products = orders.filter(F.col("customer_id") == 1).select("product_id").distinct()
(orders.filter(F.col("customer_id") != 1)
       .join(aziz_products, "product_id", "left_semi")
       .join(customers, "customer_id")
       .select("name").distinct()
       .orderBy("name")
       .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;Show the pairs of orders by the same customer placed at most 20 days apart: customer_id, first_order, ...</summary>

```python
o1, o2 = orders.alias("o1"), orders.alias("o2")
(o1.join(o2, (F.col("o1.customer_id") == F.col("o2.customer_id"))
             & (F.col("o1.order_id") < F.col("o2.order_id"))
             & (F.datediff(F.col("o2.order_date"), F.col("o1.order_date")) <= 20))
   .select(F.col("o1.customer_id").alias("customer_id"),
           F.col("o1.order_id").alias("first_order"),
           F.col("o2.order_id").alias("second_order"),
           F.datediff(F.col("o2.order_date"), F.col("o1.order_date")).alias("days_between"))
   .orderBy("customer_id", "first_order")
   .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;For each delivered order, find the promotion of its product's category that was running on the order date. ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .join(promotions, (products.category == promotions.category)
                         & F.col("order_date").between(F.col("start_date"), F.col("end_date")))
       .select("order_id", "product_name", "promo_id", "discount_pct")
       .orderBy("order_id")
       .show())
```

</details>

<details>
<summary><b>B7</b> &nbsp;Revenue after promotions: for every delivered order, revenue = quantity × price × (1 − discount_pct / 100), ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .join(promotions, (products.category == promotions.category)
                         & F.col("order_date").between(F.col("start_date"), F.col("end_date")), "left")
       .select("order_id",
               F.round(F.col("quantity") * F.col("price") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0)) / 100), 2)
                .alias("revenue"))
       .orderBy("order_id")
       .show())
```

</details>

<details>
<summary><b>B8</b> &nbsp;Make a table with every combination of customers 1 to 3 and every product category, with the number of orders ...</summary>

```python
combos = customers.filter(F.col("customer_id") <= 3).select("customer_id").crossJoin(products.select("category").distinct())
counts = orders.join(products, "product_id").groupBy("customer_id", "category").agg(F.count("*").alias("orders"))
(combos.join(counts, ["customer_id", "category"], "left")
       .select("customer_id", "category", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
       .orderBy("customer_id", "category")
       .show())
```

</details>

<details>
<summary><b>B9</b> &nbsp;Which orders point to a product that is not in products? Show order_id and product_id.</summary>

```python
orders.join(products, "product_id", "left_anti").select("order_id", "product_id").show()
```

</details>

<details>
<summary><b>B10</b> &nbsp;Show the revenue of delivered orders per city and category (customers without a city drop out). Columns: ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(customers, "customer_id")
       .join(products, "product_id")
       .filter(F.col("city").isNotNull())
       .groupBy("city", "category")
       .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue"))
       .orderBy("city", "category")
       .show())
```

</details>

<details>
<summary><b>B11</b> &nbsp;Which customers ordered in all three months (January, February and March)? Show their names.</summary>

```python
(orders.groupBy("customer_id")
       .agg(F.count_distinct(F.month("order_date")).alias("months"))
       .filter(F.col("months") == 3)
       .join(customers, "customer_id")
       .select("name")
       .orderBy("name")
       .show())
```

</details>

<details>
<summary><b>B12</b> &nbsp;Which customers had all of their orders delivered (and at least one order)? Show their names.</summary>

```python
not_delivered = orders.filter(F.col("status") != "delivered").select("customer_id")
(customers.join(orders, "customer_id", "left_semi")
          .join(not_delivered, "customer_id", "left_anti")
          .select("name")
          .orderBy("name")
          .show())
```

</details>

<details>
<summary><b>B13</b> &nbsp;For each order placed in March, show order_id and the customer's name, or Guest when the order has no ...</summary>

```python
(orders.filter(F.month("order_date") == 3)
       .join(customers, "customer_id", "left")
       .select("order_id", F.coalesce(F.col("name"), F.lit("Guest")).alias("customer"))
       .orderBy("order_id")
       .show())
```

</details>